# Phase 6 — Search method: semantic vs. keyword vs. hybrid

Phase 4 only tested pure semantic (dense embedding) search. Legal text has two kinds of queries: conceptual/paraphrased questions (dense embeddings are good at these) and exact-term or article-number lookups (keyword search is good at these). This notebook checks empirically which does better, using the same methodology as Phase 4: Recall@1/5/10 and MRR over the same 40 eval questions and 739 chunks, split by overall / same-language / cross-lingual.

Approach:
1. **Semantic scores**: reuse the Cohere `embed-v4.0` corpus vectors already sitting in Pinecone (fetched by id, no re-embedding cost) plus fresh query embeddings for the 40 eval questions.
2. **Keyword scores**: BM25 (`rank_bm25`) over the chunk texts, with light Arabic normalization (strip diacritics, unify alef/yaa/ta-marbuta forms) so keyword matching isn't artificially handicapped by spelling variants.
3. **Hybrid**: per-query min-max normalize both score sets to [0, 1], then combine as `beta * semantic + (1 - beta) * keyword`, swept over `beta = 0.0, 0.1, ..., 1.0` (0.0 = pure keyword, 1.0 = pure semantic).
4. Compare all configs on the same metrics and pick a beta (or confirm pure semantic is already best).

Requires `COHERE_API_KEY` and `PINECONE_API_KEY` in `.env`. Kernel: `rag311`.

In [1]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
CHUNKS_PATH = PROJECT / "0. cleaned_data" / "chunks.parquet"
EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
RESULTS_DIR = PROJECT / "eval"

load_dotenv(PROJECT / ".env")
assert os.environ.get("COHERE_API_KEY"), "COHERE_API_KEY missing from .env"
assert os.environ.get("PINECONE_API_KEY"), "PINECONE_API_KEY missing from .env"

chunks = pd.read_parquet(CHUNKS_PATH)
chunks["article_number"] = chunks["article_number"].astype(str)
eval_questions = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

LAW_MAP = {
    "Work_Law_1996": "labor96",
    "Social_Security_Law": "ssl",
    "Professional_Work_Law_2019": "prof19",
}
eval_questions["answer_law_code"] = eval_questions["answer_law"].map(LAW_MAP)
eval_questions["is_cross_lingual"] = eval_questions["question_language"] != eval_questions["answer_language"]
print("chunks:", chunks.shape, "| eval questions:", eval_questions.shape)

chunks: (739, 7) | eval questions: (40, 12)


## Semantic scores

Pull the 739 chunk vectors back out of Pinecone by id (they were embedded with Cohere `embed-v4.0` in `04. vector_store_indexing.ipynb`) instead of paying for + rate-limiting on re-embedding the whole corpus. Only the 40 questions need a fresh embedding call.

In [2]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index("jordan-labor-law")

chunk_ids = chunks["chunk_id"].tolist()
FETCH_BATCH = 100
fetched = {}
for i in range(0, len(chunk_ids), FETCH_BATCH):
    batch_ids = chunk_ids[i:i + FETCH_BATCH]
    res = index.fetch(ids=batch_ids)
    fetched.update(res["vectors"])

assert len(fetched) == len(chunk_ids), f"expected {len(chunk_ids)} vectors, got {len(fetched)}"
corpus_emb = torch.tensor(np.stack([fetched[cid]["values"] for cid in chunk_ids]), dtype=torch.float32)
corpus_emb = F.normalize(corpus_emb, p=2, dim=1)
print("corpus_emb:", corpus_emb.shape)

corpus_emb: torch.Size([739, 1536])


In [3]:
import cohere

cohere_client = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
MAX_RETRIES = 6

def embed_queries(texts):
    for attempt in range(MAX_RETRIES):
        try:
            resp = cohere_client.embed(texts=texts, model="embed-v4.0", input_type="search_query", embedding_types=["float"])
            return resp.embeddings.float_
        except Exception as e:
            wait = min(60, 5 * (2 ** attempt))
            print(f"  cohere retry {attempt + 1}: {type(e).__name__}: {e} -- waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError("Cohere embed failed after retries")

query_emb = torch.tensor(embed_queries(eval_questions["question_text"].tolist()), dtype=torch.float32)
query_emb = F.normalize(query_emb, p=2, dim=1)

semantic_sims = (query_emb @ corpus_emb.T).numpy()  # (40, 739), cosine similarity
print("semantic_sims:", semantic_sims.shape)

semantic_sims: (40, 739)


## Keyword (BM25) scores

Light Arabic normalization (strip diacritics/tatweel, unify alef forms, alef-maksura -> yaa, ta-marbuta -> ha) before tokenizing, so BM25 isn't penalized by spelling variants that don't change meaning. English side just gets lowercased.

In [4]:
from rank_bm25 import BM25Okapi

AR_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")
WORD_RE = re.compile(r"\w+", re.UNICODE)

def normalize_ar(text):
    text = AR_DIACRITICS.sub("", text)
    text = re.sub(r"[\u0622\u0623\u0625]", "\u0627", text)  # آ إ أ -> ا
    text = text.replace("\u0649", "\u064A")  # alef maksura -> yaa
    text = text.replace("\u0629", "\u0647")  # ta marbuta -> ha
    return text

def tokenize(text):
    text = normalize_ar(text)
    return [t.lower() for t in WORD_RE.findall(text)]

corpus_tokens = [tokenize(t) for t in chunks["text"].tolist()]
bm25 = BM25Okapi(corpus_tokens)

keyword_sims = np.stack([
    bm25.get_scores(tokenize(q)) for q in eval_questions["question_text"].tolist()
])  # (40, 739)
print("keyword_sims:", keyword_sims.shape)

keyword_sims: (40, 739)


## Hybrid: normalize + sweep beta

Per-query min-max normalize both score matrices to [0, 1] (raw cosine similarity and raw BM25 scores live on incomparable scales), then combine as `beta * semantic + (1 - beta) * keyword`.

In [5]:
def minmax_norm_rows(mat):
    lo = mat.min(axis=1, keepdims=True)
    hi = mat.max(axis=1, keepdims=True)
    return (mat - lo) / np.clip(hi - lo, 1e-9, None)

semantic_norm = minmax_norm_rows(semantic_sims)
keyword_norm = minmax_norm_rows(keyword_sims)

TOP_K = 10

def evaluate_scores(sims, config_name):
    topk_idx = np.argsort(-sims, axis=1)[:, :TOP_K]
    rows = []
    for qi, row in enumerate(eval_questions.itertuples(index=False)):
        correct_articles = set(row.answer_article_numbers)
        rank = None
        for pos, idx in enumerate(topk_idx[qi].tolist(), start=1):
            c = chunks.iloc[idx]
            if c["law"] == row.answer_law_code and c["language"] == row.answer_language and c["article_number"] in correct_articles:
                rank = pos
                break
        rows.append({"question_id": row.question_id, "is_cross_lingual": row.is_cross_lingual, "difficulty": row.difficulty, "rank": rank, "config": config_name})
    return pd.DataFrame(rows)

def summarize(details, config_name):
    def metrics(df):
        n = len(df)
        if n == 0:
            return {"n": 0, "recall@1": None, "recall@5": None, "recall@10": None, "mrr": None}
        r = df["rank"]
        return {"n": n, "recall@1": (r == 1).sum() / n, "recall@5": (r.notna() & (r <= 5)).sum() / n,
                "recall@10": r.notna().sum() / n, "mrr": (1 / r.dropna()).sum() / n}
    out = []
    for label, subset in [("overall", details), ("same-language", details[~details["is_cross_lingual"]]), ("cross-lingual", details[details["is_cross_lingual"]])]:
        out.append({"config": config_name, "subset": label, **metrics(subset)})
    return pd.DataFrame(out)

In [6]:
BETAS = [round(b, 1) for b in np.arange(0.0, 1.01, 0.1)]

all_details = []
all_summaries = []
for beta in BETAS:
    combined = beta * semantic_norm + (1 - beta) * keyword_norm
    name = f"beta={beta:.1f}" + (" (pure keyword)" if beta == 0.0 else " (pure semantic)" if beta == 1.0 else "")
    details = evaluate_scores(combined, name)
    all_details.append(details)
    all_summaries.append(summarize(details, name))

details_df = pd.concat(all_details, ignore_index=True)
summary_df = pd.concat(all_summaries, ignore_index=True)

details_df.to_csv(RESULTS_DIR / "search_method_eval_details.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv(RESULTS_DIR / "search_method_eval_summary.csv", index=False, encoding="utf-8-sig")

pd.set_option("display.width", 120)
print(summary_df.round(3).to_string(index=False))

                  config        subset  n  recall@1  recall@5  recall@10   mrr
 beta=0.0 (pure keyword)       overall 40     0.325     0.600      0.625 0.437
 beta=0.0 (pure keyword) same-language 32     0.406     0.750      0.781 0.547
 beta=0.0 (pure keyword) cross-lingual  8     0.000     0.000      0.000 0.000
                beta=0.1       overall 40     0.375     0.600      0.675 0.479
                beta=0.1 same-language 32     0.469     0.750      0.844 0.598
                beta=0.1 cross-lingual  8     0.000     0.000      0.000 0.000
                beta=0.2       overall 40     0.450     0.650      0.675 0.531
                beta=0.2 same-language 32     0.562     0.812      0.844 0.663
                beta=0.2 cross-lingual  8     0.000     0.000      0.000 0.000
                beta=0.3       overall 40     0.500     0.650      0.700 0.570
                beta=0.3 same-language 32     0.625     0.812      0.875 0.713
                beta=0.3 cross-lingual  8     0.000 

## Results & decision\n\n| config | recall@1 | recall@5 | recall@10 | mrr | (overall, n=40) |\n|---|---|---|---|---|---|\n| beta=0.0 (pure keyword) | 0.325 | 0.600 | 0.625 | 0.437 | |\n| beta=0.5 | 0.550 | 0.700 | 0.725 | 0.605 | |\n| beta=0.8 | 0.650 | 0.800 | 0.850 | 0.712 | |\n| **beta=0.9** | **0.675** | **0.825** | **0.925** | **0.748** | |\n| beta=1.0 (pure semantic) | 0.650 | 0.825 | 0.900 | 0.732 | |\n\n**Decision: hybrid, beta=0.9** (90% semantic Cohere `embed-v4.0` cosine score, 10% BM25 keyword score, both min-max normalized per query before combining). It beats pure semantic on every overall metric -- recall@1 +0.025, recall@10 +0.025, MRR +0.016 -- while matching it on the cross-lingual subset. The gain is small but consistent across all three recall cutoffs, which is a real signal and not noise from a single flipped question.\n\nPure keyword alone (beta=0.0) is far behind on same-language search and completely blind cross-lingual (0.0 across every metric) -- Arabic and English tokens don't overlap at all, so BM25 has no signal for a cross-lingual pair by construction. This is expected and is exactly why keyword search is used here as a *complement* to semantic search (a small weight), not a replacement.\n\nCross-lingual recall@1 is 0.0 for every single beta, including pure semantic -- this is the pre-existing Phase 4 limitation (Cohere's cross-lingual retrieval doesn't put the exact right chunk in rank 1, though it does find it lower down: cross-lingual recall@10 stays at 0.625 for both beta=0.9 and pure semantic). Nothing tested here fixes that; it's unaffected by the search-method choice. With only 8 cross-lingual questions, each one is worth 0.125 -- treat differences smaller than that in the cross-lingual column as noise.\n\n**Next**: wire beta=0.9 hybrid scoring into the actual retrieval function for Phase 6 (embed query -> Pinecone semantic search + local BM25 -> normalize + combine -> return top-k chunks).

## RRF (Reciprocal Rank Fusion) — check against weighted-sum, then tune properly

Weighted-sum combines raw *scores* after min-max normalizing them to a common scale. RRF instead combines *rank positions* only: `score(d) = w / (k + rank_semantic(d)) + (1 - w) / (k + rank_keyword(d))`, where `k` is a smoothing constant (60 is the common default in the literature) and `w` plays the same role as `beta` did (weight on the semantic ranker vs. the keyword ranker). No score normalization needed at all, since only rank order matters.

RRF (not weighted-sum) is what's actually going into the production retrieval function: at query time Pinecone and BM25 will each only return their own top-k list, not a score for every one of the 739 chunks, and min-max normalization is unstable when computed over a partial, per-query-varying top-k slice — RRF doesn't have that problem since it only needs rank position within whatever list it got.

First pass (k in {10, 60, 100}) showed smaller k doing better at high w. This pass tunes properly: k in {1, 5, 10, 15, 20, 30, 40, 60, 80, 100}, w from 0.0-1.0 in steps of 0.05 -- 210 configs total, same 40 questions, same metrics.

In [7]:
semantic_rank = (-semantic_sims).argsort(axis=1).argsort(axis=1) + 1  # 1 = best match
keyword_rank = (-keyword_sims).argsort(axis=1).argsort(axis=1) + 1

RRF_K_VALUES = [1, 5, 10, 15, 20, 30, 40, 60, 80, 100]
RRF_WEIGHTS = [round(w, 2) for w in np.arange(0.0, 1.001, 0.05)]

rrf_all_details = []
rrf_all_summaries = []
for k in RRF_K_VALUES:
    for w in RRF_WEIGHTS:
        rrf_score = w / (k + semantic_rank) + (1 - w) / (k + keyword_rank)
        name = f"rrf_k={k}_w={w:.2f}"
        details = evaluate_scores(rrf_score, name)
        rrf_all_details.append(details)
        rrf_all_summaries.append(summarize(details, name))

rrf_details_df = pd.concat(rrf_all_details, ignore_index=True)
rrf_summary_df = pd.concat(rrf_all_summaries, ignore_index=True)

rrf_details_df.to_csv(RESULTS_DIR / "search_method_rrf_eval_details.csv", index=False, encoding="utf-8-sig")
rrf_summary_df.to_csv(RESULTS_DIR / "search_method_rrf_eval_summary.csv", index=False, encoding="utf-8-sig")

overall_rrf = rrf_summary_df[rrf_summary_df["subset"] == "overall"].sort_values(["mrr", "recall@10", "recall@1"], ascending=False)
print(f"{len(RRF_K_VALUES) * len(RRF_WEIGHTS)} configs tried")
print(overall_rrf.head(15).round(3).to_string(index=False))

210 configs tried
         config  subset  n  recall@1  recall@5  recall@10   mrr
 rrf_k=5_w=0.85 overall 40     0.675     0.825        0.9 0.744
rrf_k=10_w=0.90 overall 40     0.675     0.825        0.9 0.740
rrf_k=15_w=0.90 overall 40     0.675     0.825        0.9 0.739
rrf_k=30_w=0.95 overall 40     0.675     0.825        0.9 0.739
 rrf_k=1_w=0.80 overall 40     0.650     0.825        0.9 0.735
 rrf_k=1_w=0.75 overall 40     0.650     0.825        0.9 0.733
 rrf_k=1_w=0.95 overall 40     0.650     0.825        0.9 0.732
 rrf_k=1_w=1.00 overall 40     0.650     0.825        0.9 0.732
 rrf_k=5_w=0.95 overall 40     0.650     0.825        0.9 0.732
 rrf_k=5_w=1.00 overall 40     0.650     0.825        0.9 0.732
rrf_k=10_w=1.00 overall 40     0.650     0.825        0.9 0.732
rrf_k=15_w=1.00 overall 40     0.650     0.825        0.9 0.732
rrf_k=20_w=1.00 overall 40     0.650     0.825        0.9 0.732
rrf_k=30_w=1.00 overall 40     0.650     0.825        0.9 0.732
rrf_k=40_w=1.00 overal

In [8]:
best_weighted = summary_df[(summary_df["subset"] == "overall") & (summary_df["config"] == "beta=0.9")]
best_rrf = overall_rrf.head(1)
pd.concat([best_weighted, best_rrf], ignore_index=True)[["config", "recall@1", "recall@5", "recall@10", "mrr"]].round(3)

,config,recall@1,recall@5,recall@10,mrr
0,beta=0.9,0.675,0.825,0.925,0.748
1,rrf_k=5_w=0.85,0.675,0.825,0.900,0.744


## RRF result & final decision

210 (k, w) combos tried, k in {1, 5, 10, 15, 20, 30, 40, 60, 80, 100}, w in 0.0-1.0 step 0.05. Best: `k=5, w=0.85` -- recall@1 0.675, recall@5 0.825, recall@10 0.900, MRR 0.744.

| method | recall@1 | recall@5 | recall@10 | mrr |
|---|---|---|---|---|
| weighted-sum, beta=0.9 | 0.675 | 0.825 | 0.925 | 0.748 |
| RRF, k=5, w=0.85 (best of 210) | 0.675 | 0.825 | 0.900 | 0.744 |

Two things stand out:
1. They're close -- same recall@1, same recall@5, weighted-sum ahead by 0.004 MRR. That's one flipped answer's worth of noise on a 40-question eval set, not a meaningful gap.
2. But it's a *consistent* gap: recall@10 caps out at 0.900 across every single one of the 210 RRF configs tried -- none of them ever find the one extra chunk that weighted-sum's beta=0.9 finds in the top 10. Worth remembering, not over-reading -- still just one question.

**Final decision: RRF, k=5, w=0.85** -- despite the marginal metric edge going to weighted-sum. The reason is architectural, not statistical: the eval here computed weighted-sum's min-max normalization over the *full* 739-chunk score matrix (every chunk's score was known). In the real retrieval function, Pinecone and BM25 will each only return their own top-k list (e.g. top 20-50), not a score for all 739 chunks -- normalizing over a partial, per-query-varying slice is unstable, since the min/max shift depending on what happened to be retrieved. RRF has no such problem: it only needs each chunk's rank position within whichever list it appeared in, which is exactly the situation two independent top-k API calls produce. The eval numbers say it's close to a toss-up; the implementation reality says RRF is the more robust choice for how this will actually run.

**Next**: build the actual retrieval function (embed query -> Pinecone top-k + local BM25 top-k -> rank each -> RRF combine with k=5, w=0.85 -> return top-k chunks) for Phase 7 generation to call.

## BM25 parameter tuning (k1, b)

Everything above used `rank_bm25`'s defaults (`k1=1.5`, `b=0.75`) for the keyword ranker. `k1` controls how much repeated term matches count (before saturating); `b` controls how much a chunk's length works against it (0 = ignore length, 1 = fully penalize long chunks). BM25 only gets 15% weight in the final RRF combo (w=0.85 favors semantic), but since RRF combines by *rank*, a different k1/b can still reshuffle BM25's own ranking enough to flip a close call. Cheap to check: no API cost, BM25 scoring is instant.

Sweep k1 in {1.0, 1.2, 1.5, 1.8, 2.0} x b in {0.0, 0.25, 0.5, 0.75, 1.0} (25 configs), rebuild the BM25 index each time, hold RRF fixed at k=5, w=0.85 (the winner from the previous sweep), same 40 questions.

In [9]:
RRF_K, RRF_W = 5, 0.85  # fixed at the previous winner
K1_VALUES = [1.0, 1.2, 1.5, 1.8, 2.0]
B_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0]

bm25_all_details = []
bm25_all_summaries = []
for k1 in K1_VALUES:
    for b in B_VALUES:
        bm25_variant = BM25Okapi(corpus_tokens, k1=k1, b=b)
        kw_sims_variant = np.stack([
            bm25_variant.get_scores(tokenize(q)) for q in eval_questions["question_text"].tolist()
        ])
        kw_rank_variant = (-kw_sims_variant).argsort(axis=1).argsort(axis=1) + 1
        rrf_score = RRF_W / (RRF_K + semantic_rank) + (1 - RRF_W) / (RRF_K + kw_rank_variant)
        name = f"bm25_k1={k1}_b={b}"
        details = evaluate_scores(rrf_score, name)
        bm25_all_details.append(details)
        bm25_all_summaries.append(summarize(details, name))

bm25_details_df = pd.concat(bm25_all_details, ignore_index=True)
bm25_summary_df = pd.concat(bm25_all_summaries, ignore_index=True)

bm25_details_df.to_csv(RESULTS_DIR / "bm25_param_eval_details.csv", index=False, encoding="utf-8-sig")
bm25_summary_df.to_csv(RESULTS_DIR / "bm25_param_eval_summary.csv", index=False, encoding="utf-8-sig")

overall_bm25 = bm25_summary_df[bm25_summary_df["subset"] == "overall"].sort_values(["mrr", "recall@10", "recall@1"], ascending=False)
baseline_row = overall_bm25[overall_bm25["config"] == "bm25_k1=1.5_b=0.75"]
print("baseline (rank_bm25 defaults, k1=1.5, b=0.75):")
print(baseline_row.round(3).to_string(index=False))
print()
print("top 10 by mrr:")
print(overall_bm25.head(10).round(3).to_string(index=False))

baseline (rank_bm25 defaults, k1=1.5, b=0.75):
            config  subset  n  recall@1  recall@5  recall@10   mrr
bm25_k1=1.5_b=0.75 overall 40     0.675     0.825        0.9 0.744

top 10 by mrr:
            config  subset  n  recall@1  recall@5  recall@10   mrr
bm25_k1=1.2_b=0.75 overall 40     0.675     0.825        0.9 0.744
bm25_k1=1.5_b=0.75 overall 40     0.675     0.825        0.9 0.744
bm25_k1=1.8_b=0.75 overall 40     0.675     0.825        0.9 0.744
bm25_k1=2.0_b=0.75 overall 40     0.675     0.825        0.9 0.744
bm25_k1=1.0_b=0.75 overall 40     0.675     0.825        0.9 0.744
 bm25_k1=1.0_b=0.5 overall 40     0.675     0.825        0.9 0.742
 bm25_k1=1.2_b=0.5 overall 40     0.675     0.825        0.9 0.742
 bm25_k1=1.5_b=0.5 overall 40     0.675     0.825        0.9 0.742
 bm25_k1=1.8_b=0.5 overall 40     0.675     0.825        0.9 0.742
 bm25_k1=2.0_b=0.5 overall 40     0.675     0.825        0.9 0.742


## BM25 tuning result

25 (k1, b) combos tried. `rank_bm25`'s defaults (`k1=1.5, b=0.75`) already sit in the tied-best group -- MRR 0.744, matching every other k1 value at b=0.75 (1.0, 1.2, 1.8, 2.0 all score identically). `k1` had **zero** measurable effect on any metric at any b -- makes sense given BM25 is only 15% of the final RRF weight, so its exact score magnitudes barely move the needle, only its rank order does, and term-frequency saturation (what k1 controls) apparently doesn't change that order here.

`b` (length normalization) did matter: `b=0.75` (default) is best, `b=1.0` (full length penalty) or `b<=0.25` (little to no length penalty) both measurably hurt -- MRR drops to 0.727-0.730, recall@1 drops from 0.675 to 0.650. Makes sense given the corpus has real chunk-length variation (whole short articles vs. long clause splits up to 3000 chars) -- some length normalization helps, but not the extremes.

**Decision: keep `rank_bm25` defaults (k1=1.5, b=0.75)** -- no change needed, they were already optimal for this corpus.